# 13 · An end-to-end 3D network on harmonized volumes (plan 7o, G4)

**Project:** Acquisition shortcuts in public cancer imaging (Hamza El Massaoudy)
**Notebook written:** 7 Nov 2026 · **Needs:** `results/12_image_experiments/` (harmonized volumes and labels) · **Plan:**
section **7o**, G4 (`protocol/analysis_plan_7o.md`), written before any network was trained.

**What this notebook does, in plain words**

The earlier image results come from frozen foundation models with a linear probe. Here a 3D ResNet-18 is trained
end to end on TCGA lung (adenocarcinoma versus squamous cell carcinoma), once on the full image and once on the
background only (body replaced by air), using the harmonized 2 × 2 × 5 mm volumes of notebook 12. It is evaluated by
5-fold cross-validation in TCGA (three training seeds per fold) and then, trained on all TCGA patients (three seeds),
applied unchanged to CPTAC lung and the three independent lung cohorts, with plan 7o's reading rule.

**Rehearsal first.** With `RUN_MODE = "both"`, the network is first trained on **shuffled labels** (one seed) and must
look like chance before the real labels are used.

**How to run:** in Colab choose *Runtime → Change runtime type → A100 GPU (or L4), High-RAM*, then *Runtime → Run all*.
Expect about 2–4 hours. **If Colab disconnects, run all again:** every finished training is kept on Drive. Outputs go to
`results/13_cnn/`. The last cell stops the runtime.

In [ ]:
# 1. Setup: Drive, folders, GPU
import os, sys, json, time, hashlib, datetime, platform, subprocess, warnings
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
RUN_MODE = 'both'  # @param ["both", "rehearsal", "real"]
if IN_COLAB:
    PROJECT = Path('/content/drive/MyDrive/Acquisition-Shortcut-Audit')
else:  # local code tests only
    PROJECT = Path(os.environ['ASA_PROJECT_DIR'])
    RUN_MODE = os.environ.get('ASA_TEST_RUN_MODE', 'rehearsal')
IN12 = PROJECT / 'results' / '12_image_experiments'
OUT_ROOT = PROJECT / 'results' / '13_cnn'
for m in ('rehearsal', 'real'):
    assert (IN12 / m / 'run_manifest.json').exists(), f'{IN12 / m} is missing: run notebook 12 first.'
(OUT_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)
RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
print('Outputs to:', OUT_ROOT, '| run mode:', RUN_MODE, '| started', RUN_STARTED, 'UTC')

In [ ]:
# 2. Packages (MONAI pinned as in notebooks 05-12)
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai==1.6.1'], check=True)
import numpy as np
import pandas as pd
import scipy
from scipy.stats import rankdata
import sklearn
from sklearn.model_selection import StratifiedKFold
import torch
import monai
from monai.networks.nets import resnet18
from monai.transforms import Compose, RandFlip, RandRotate
warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def show(df, n=30):
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
            'scikit-learn': sklearn.__version__, 'torch': torch.__version__, 'monai': monai.__version__,
            'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none'}
print(VERSIONS)

In [ ]:
# 3. Settings (plan 7o, G4; fixed before any training)
SEED = 2026
N_FOLDS = 5
SEEDS_REAL, SEEDS_REHEARSAL = (0, 1, 2), (0,)
EPOCHS, LR, WEIGHT_DECAY, BATCH = 60, 1e-4, 1e-2, 8
HU_RANGE = (-1000.0, 1000.0)            # clipped and scaled to 0-1 (Merlin's window)
N_BOOT, N_PERM_STRICT = 2000, 99999
SOURCE = 'lung histology (LUAD vs LUSC)'
LUNG_TARGETS = ('CPTAC lung', 'NSCLC Radiogenomics', 'NSCLC-Radiomics', 'Lung-PET-CT-Dx')
INDEPENDENT = ('NSCLC Radiogenomics', 'NSCLC-Radiomics', 'Lung-PET-CT-Dx')
INPUTS = ('full', 'background')
GATE_RANGE = (0.35, 0.65)
if not IN_COLAB:                         # local code tests only
    EPOCHS = int(os.environ.get('ASA_TEST_EPOCHS', 1)); N_BOOT = int(os.environ.get('ASA_TEST_N_BOOT', 50))
    N_PERM_STRICT = int(os.environ.get('ASA_TEST_N_PERM', 99))
SETTINGS = {k: v for k, v in dict(SEED=SEED, N_FOLDS=N_FOLDS, SEEDS_REAL=SEEDS_REAL, SEEDS_REHEARSAL=SEEDS_REHEARSAL,
                                   EPOCHS=EPOCHS, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, BATCH=BATCH, HU_RANGE=HU_RANGE,
                                   N_BOOT=N_BOOT, N_PERM_STRICT=N_PERM_STRICT).items()}
SETTINGS_KEY = hashlib.md5(json.dumps(SETTINGS, sort_keys=True, default=str).encode()).hexdigest()[:12]
print(SETTINGS, SETTINGS_KEY)

In [ ]:
# 4. Labels (saved by notebook 12) and the harmonized volumes
def labels_of(mode):
    lab = pd.read_csv(IN12 / mode / 'dataset_labels.csv')
    lab = lab[lab.dataset.isin((SOURCE,) + LUNG_TARGETS)]
    have = {p.stem for p in (IN12 / 'cache' / 'cnn').glob('*.npz')}
    missing = lab[~lab.key.isin(have)]
    return lab[lab.key.isin(have)].reset_index(drop=True), missing

lab_real, miss_real = labels_of('real')
print(lab_real.groupby(['dataset', 'class']).size().to_string())
print('patients without a harmonized volume (left out):', len(miss_real))
VOL = {}
for k in lab_real.key:
    with np.load(IN12 / 'cache' / 'cnn' / f'{k}.npz') as z:
        VOL[k] = {'full': z['full'], 'background': z['background']}
print(f'{len(VOL)} volumes in memory, shape {next(iter(VOL.values()))["full"].shape}')

In [ ]:
# 5. The network, training and prediction
AUG = Compose([RandFlip(prob=0.5, spatial_axis=2), RandRotate(range_x=np.deg2rad(10), prob=0.5, keep_size=True,
                                                               padding_mode='border')])

def to_tensor(a):
    x = (np.clip(a.astype(np.float32), *HU_RANGE) - HU_RANGE[0]) / (HU_RANGE[1] - HU_RANGE[0])
    return torch.from_numpy(x)[None]           # (1, z, y, x)

class Vols(torch.utils.data.Dataset):
    def __init__(self, keys, y, inp, train):
        self.keys, self.y, self.inp, self.train = list(keys), np.asarray(y, np.float32), inp, train
    def __len__(self):
        return len(self.keys)
    def __getitem__(self, i):
        x = to_tensor(VOL[self.keys[i]][self.inp])
        if self.train:
            x = AUG(x)
            x = x.as_tensor() if hasattr(x, 'as_tensor') else x
        return x.float(), torch.tensor(self.y[i])

def train_model(keys, y, inp, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    AUG.set_random_state(seed=seed)
    net = resnet18(spatial_dims=3, n_input_channels=1, num_classes=1).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    pos = float(np.mean(y)); w = torch.tensor((1 - pos) / max(pos, 1e-6), device=DEVICE)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=w)
    g = torch.Generator(); g.manual_seed(seed)
    dl = torch.utils.data.DataLoader(Vols(keys, y, inp, True), batch_size=BATCH, shuffle=True, generator=g,
                                     num_workers=2 if IN_COLAB else 0, drop_last=False)
    scaler = torch.amp.GradScaler(enabled=DEVICE == 'cuda')
    net.train()
    for ep in range(EPOCHS):
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', enabled=DEVICE == 'cuda'):
                loss = loss_fn(net(xb).squeeze(1), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    return net.eval()

@torch.no_grad()
def predict(net, keys, inp):
    out = []
    for i in range(0, len(keys), 16):
        xb = torch.stack([to_tensor(VOL[k][inp]) for k in keys[i:i + 16]]).to(DEVICE)
        out.append(net(xb).squeeze(1).float().cpu().numpy())
    return np.concatenate(out)

def checkpointed(name, fn):
    """Each training's predictions are saved on Drive; a rerun loads them instead of training again."""
    p = OUT_ROOT / 'checkpoints' / f'{name}__{SETTINGS_KEY}.npz'
    if p.exists():
        with np.load(p) as z:
            return {k: z[k] for k in z.files}
    res = fn()
    tmp = p.with_name(p.stem + '.tmp.npz'); np.savez(tmp, **res); tmp.replace(p)
    return res

In [ ]:
# 6. AUC, bootstrap CI and the strict test (as notebooks 05-12)
def auc_binary(y, s):
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def boot_ci(y, s):
    rng = np.random.default_rng(SEED); vals = []
    while len(vals) < N_BOOT:
        i = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[i])) == 2:
            vals.append(auc_binary(y[i], s[i]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def strict_p(yt, s):
    obs = auc_binary(yt, s); rng = np.random.default_rng(SEED)
    r = rankdata(s); n1 = int(yt.sum()); n0 = len(yt) - n1; cnt = 0
    for i in range(0, N_PERM_STRICT, 10000):
        m = min(10000, N_PERM_STRICT - i)
        P = np.argsort(rng.random((m, len(yt))), axis=1) < n1
        cnt += int(((((P * r).sum(axis=1) - n1 * (n1 + 1) / 2) / (n1 * n0)) >= obs - 1e-12).sum())
    return (1 + cnt) / (1 + N_PERM_STRICT)

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); out = np.empty(m); run = 0.0
    for rank, j in enumerate(o):
        run = max(run, min(1.0, (m - rank) * p[j])); out[j] = run
    return out

In [ ]:
# 7. Run one mode: 5-fold cross-validation in TCGA, then transfer to the four lung cohorts
def run_mode13(mode):
    out = OUT_ROOT / mode; out.mkdir(parents=True, exist_ok=True)
    lab, miss = labels_of(mode)
    seeds = SEEDS_REHEARSAL if mode == 'rehearsal' else SEEDS_REAL
    src = lab[lab.dataset == SOURCE].reset_index(drop=True)
    ks, ys = src.key.to_numpy(), src.y.to_numpy()
    rows_int, rows_ext, scores = [], [], []
    folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(ks, ys))
    for inp in INPUTS:
        oof = np.zeros((len(seeds), len(ks)))
        for f, (tr, te) in enumerate(folds):
            for si, sd in enumerate(seeds):
                r = checkpointed(f'{mode}__cv__{inp}__fold{f}__seed{sd}',
                                 lambda: {'p': predict(train_model(ks[tr], ys[tr], inp, sd), list(ks[te]), inp)})
                oof[si, te] = r['p']
            print(f'  {mode} {inp}: fold {f + 1}/{N_FOLDS} done')
        s = oof.mean(0); a = auc_binary(ys, s); lo, hi = boot_ci(ys, s)
        rows_int.append({'input': inp, 'n': len(ys), 'auc': a, 'ci_low': lo, 'ci_high': hi, 'seeds': len(seeds)})
        ext = {t: [] for t in LUNG_TARGETS}
        for sd in seeds:
            def fit_all(sd=sd):
                net = train_model(ks, ys, inp, sd)
                return {t: predict(net, list(lab[lab.dataset == t].key), inp) for t in LUNG_TARGETS}
            r = checkpointed(f'{mode}__all__{inp}__seed{sd}', fit_all)
            for t in LUNG_TARGETS:
                ext[t].append(r[t])
        for t in LUNG_TARGETS:
            d = lab[lab.dataset == t]
            yt, st = d.y.to_numpy(), np.mean(ext[t], axis=0)
            a = auc_binary(yt, st); lo, hi = boot_ci(yt, st)
            rows_ext.append({'input': inp, 'target': t, 'n_target': len(yt), 'external_auc': a, 'ci_low': lo,
                             'ci_high': hi, 'transfers_ci': bool(lo > 0.5), 'strict_p': strict_p(yt, st)})
            scores.append(pd.DataFrame({'input': inp, 'target': t, 'key': d.key, 'y': yt, 'score': st}))
    ti, te_ = pd.DataFrame(rows_int), pd.DataFrame(rows_ext)
    te_['strict_p_holm'] = holm(te_.strict_p); te_['transfers'] = te_.transfers_ci & (te_.strict_p_holm < 0.05)
    rd = []
    for inp, g in te_.groupby('input'):
        ind = int(g[g.target.isin(INDEPENDENT)].transfers.sum()); cp = bool(g[g.target == 'CPTAC lung'].transfers.any())
        rd.append({'input': inp, 'independent_cohorts_transferred': ind, 'cptac_transferred': cp,
                   'reading': ('generalizes' if ind >= 2 else 'behaves like the background' if (cp and ind == 0)
                               else 'no transferable signal' if (ind == 0 and not cp) else 'mixed')})
    rd = pd.DataFrame(rd)
    for n, t in (('cnn_internal.csv', ti), ('cnn_transfer.csv', te_), ('cnn_reading.csv', rd)):
        t.to_csv(out / n, index=False)
    pd.concat(scores).to_csv(out / 'cnn_scores.csv.gz', index=False)
    show(ti); show(te_); show(rd)
    outputs = [{'file': p.name, 'md5': md5(p)} for p in sorted(out.glob('*.csv*'))]
    man = {'notebook': '13_cnn.ipynb', 'plan': '7o (G4)', 'mode': mode, 'labels_from': str(IN12 / mode / 'dataset_labels.csv'),
           'labels_md5': md5(IN12 / mode / 'dataset_labels.csv'), 'missing_volumes': miss.key.tolist(),
           'settings': SETTINGS, 'settings_key': SETTINGS_KEY, 'versions': VERSIONS, 'run_started_utc': RUN_STARTED,
           'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
           'in_colab': IN_COLAB, 'outputs': outputs}
    (out / 'run_manifest.json').write_text(json.dumps(man, indent=2, default=str))
    return ti, te_, rd

modes = {'both': ['rehearsal', 'real'], 'rehearsal': ['rehearsal'], 'real': ['real']}[RUN_MODE]
results13 = {}
for mode in modes:
    if mode == 'real' and 'rehearsal' in results13:
        ti_r = results13['rehearsal'][0]
        if not ti_r.auc.between(*GATE_RANGE).all():
            print(f'GATE NOT PASSED: a rehearsal AUC is outside {GATE_RANGE}: {ti_r.auc.round(3).tolist()}; '
                  'the real-label pass was NOT run.')
            break
        print('Rehearsal gate passed: the network on shuffled labels looks like chance.')
    results13[mode] = run_mode13(mode)

In [ ]:
# 8. Stop the runtime once everything above has finished
if IN_COLAB:
    from google.colab import runtime
    drive.flush_and_unmount()
    print('Drive synced. Stopping the runtime now.')
    time.sleep(5)
    runtime.unassign()